# Exporting in models

This tutorial provides inspiration and potentially useful code for exporting data in a `pp.Model` based simulation for visualization in e.g. [ParaView](https://www.paraview.org/). While the PorePy [model class](./single_phase_flow.ipynb) and the [exporter](./exporter.ipynb) are introduced elsewhere, this tutorial provides some detail on how to combine them. To this end, we will choose the transient model for mixed-dimensional poroelasticity with fracture deformation and adjust it to make sure the solution is exported according to our requirements, i.e. at the right stages of the simulation and that we export the right fields/variables. The tutorial consists of two parts. The first covers standard usage and the second covers more advanced usage related to debugging. 

We start with a very simple case, indicating which methods could be overwritten to specify which variables are exported. The starting point is the `DataSavingMixin` class, which is responsible for all things related to saving and exporting of data during a simulation. By default, it exports all primary variables as well as apertures and specific volumes, see `data_to_export`. Exporting is performed at the start of the simulation and at the end of each time step.

In [1]:
from typing import Any
import porepy as pp
import numpy as np
from porepy.models.poromechanics import Poromechanics

model_params: dict[str, Any] = {
    "folder_name": "model_exporting"
}
# The compound class Poromechanics inherits from DataSavingMixin.
model_0 = Poromechanics(model_params)
pp.run_time_dependent_model(model_0)

Since the default `set_geometry` method of `Poromechanics` (inherited from `ModelGeometry`) produces a monodimensional domain, we get data files containing pressure and displacement in the matrix domain and no tractions (no fractures are present). 


## Mixed-dimensional simulations
We now extend to the mixed-dimensional case. In the parameters, we adjust the file name to avoid overwriting the previous files. If you inspect the suffixes of the files created, you can see how the exporter deals with multiple time steps by default.

In [2]:
from porepy.applications.md_grids.model_geometries import (
    SquareDomainOrthogonalFractures,
)


class MDPoroelasticity(
    SquareDomainOrthogonalFractures,
    Poromechanics,
):
    """Combine the geometry class with the poromechanics class."""

    pass


model_params.update(
    {
        "end_time": 2,
        "file_name": "md",
    }
)
model_1 = MDPoroelasticity(model_params)
pp.run_time_dependent_model(model_1)

### Tailored exporting
Suppose we want to perform a simulation similar to above, but require more data for visualization. 
For instance, we might very reasonably want to look at the displacement jump on the fractures. 
This is not a primary variable, and thus not exported by default. 
We implement this as a second mixin class which we combine with `MDPoroelasticity`which adds a tuple containing grid, name and values to the list of data to be exported. 
For other allowed formats, see [exporter](./exporter.ipynb). 
The code below is a subset of what is contained in the class `FractureDeformationExportingModel` in the PorePy source code, but shown here for illustration. 


In [3]:
class DisplacementJumpExporting:
    def data_to_export(self):
        """Define the data to export to vtu.

        Returns:
            list: List of tuples containing the subdomain, variable name,
            and values to export.

        """
        data = super().data_to_export()
        # Add displacement jumps to the data to export.
        # While not the most straightforward way to do it, evaluating before the loop
        # increases performance and is preferable for models with many subdomains.
        sds = self.mdg.subdomains(dim=self.nd - 1)
        cell_offsets = np.cumsum([0] + [sd.num_cells for sd in sds])
        displacement_jumps = self.evaluate_and_scale(sds, "displacement_jump", "m")
        for id, sd in enumerate(sds):
            data.append(
                (
                    sd,
                    "displacement_jump",
                    displacement_jumps[cell_offsets[id] : cell_offsets[id + 1]],
                )
            )

        return data


class TailoredPoroelasticity(DisplacementJumpExporting, MDPoroelasticity):
    """Combine the exporting class with the poromechanics class."""
    pass


model_params.update(
    {
        "end_time": 2,
        "file_name": "jumps",
    }
)
model_2 = TailoredPoroelasticity(model_params)
pp.run_time_dependent_model(model_2)

# Iteration exporting for debugging
We now turn to exporting data for each iteration when solving the nonlinear system. This second part is significantly more advanced than the preceeding part and some users may want to skip it.

Exporting iterations can be quite handy when debugging or trying to make sense of why your model doesn't converge. Moreover, even when everything works as a dream, you might want to visualize how convergence is reached, for instance to distinguish between global and local effects. 

We stress that not only which variables to export but also when you wish to export them may vary between applications. In the model provided below, we export at all iterations using a separate exporter, keeping track of time step and iteration number using the vtu file suffix and collecting them using a single pvd file. The "time step" suffix of an iteration file is the sum of the iteration index and the product of the current time step index and $r$. Here $r$ is the smallest power of ten exceeding the maximum number of non-linear iterations. 

Expecting that the simulation may crash or be stopped at any point, we (over)write a pvd file each time a new vtu file is added. Alternative approaches and refinements include writing one pvd file for each time step and writing debugging files on some condition, e.g. that the iteration index exceeds some threshold.

Since the class is intended for debugging, we also export the residuals of each equation in the equation system, so that we can monitor convergence behavior in detail. This is implemented in a separate mixin class, which we combine with the iteration exporting mixin and the tailored poroelasticity model.

In [ ]:
from porepy.viz.data_saving_model_mixin import IterationExporting, ResidualExporting


class IterationCombined(IterationExporting, ResidualExporting, TailoredPoroelasticity):
    """Add iteration exporting to the tailored poroelasticity class.

    With a trivial problem, the residuals will be zero, but this serves as an example of
    how to combine the different mixins.
    """

    pass


# Update file and folder name. Both time step and iteration states are exported. The
# folder name of the latter is appended automatically with "_iterations". Thus, we
# rename the folder to keep it distinct from the above simulations.
model_params["file_name"] = "with_residuals"
model_params["folder_name"] = "model_exporting_42"
model_3 = IterationCombined(model_params)
pp.run_time_dependent_model(model_3)